In [16]:
import pandas as pd

fact = pd.read_csv("sql fact file.csv")
bus = pd.read_csv("dim_bus sql.csv")
performance = pd.read_csv("dim_performance sql.csv")
route = pd.read_csv("updated_dim_route.csv")
time = pd.read_csv("dim_time sql.csv")

In [17]:
print("Fact:", fact.shape)
print("Bus:", bus.shape)
print("Performance:", performance.shape)
print("Route:", route.shape)
print("Time:", time.shape)

Fact: (20000, 21)
Bus: (300, 6)
Performance: (20000, 7)
Route: (150, 7)
Time: (4015, 7)


In [18]:
print(fact.columns)
print(bus.columns)
print(performance.columns)
print(route.columns)
print(time.columns)

Index(['TripID', 'RouteKey', 'BusKey', 'TimeKey', 'PerfKey', 'StartTime',
       'EndTime', 'FuelConsumed', 'Distance', 'TargetTime', 'ActualTime',
       'DelayAnalysis', 'OperatingCostPerKM', 'PassengerCount',
       'EngineTemperature', 'RPM', 'FuelRate', 'IdleDuration', 'AverageSpeed',
       'BrakeEvents', 'OTPFlag'],
      dtype='object')
Index(['BusKey', 'BusID', 'BusType', 'AvgEngineTemp', 'AvgRPM', 'TotalTrips'], dtype='object')
Index(['PerfKey', 'TripID', 'OTPFlag', 'DelayAnalysis', 'AverageSpeed',
       'IdleDuration', 'BrakeEvents'],
      dtype='object')
Index(['RouteKey', 'RouteID', 'RouteName', 'Source', 'Destination',
       'RouteDistanceKM', 'RouteType'],
      dtype='object')
Index(['TimeKey', 'DateKey', 'FullDate', 'Hour', 'DayOfWeek', 'Month',
       'Quarter'],
      dtype='object')


In [19]:
print(fact.columns)
print(bus.columns)
print(performance.columns)
print(route.columns)
print(time.columns)

Index(['TripID', 'RouteKey', 'BusKey', 'TimeKey', 'PerfKey', 'StartTime',
       'EndTime', 'FuelConsumed', 'Distance', 'TargetTime', 'ActualTime',
       'DelayAnalysis', 'OperatingCostPerKM', 'PassengerCount',
       'EngineTemperature', 'RPM', 'FuelRate', 'IdleDuration', 'AverageSpeed',
       'BrakeEvents', 'OTPFlag'],
      dtype='object')
Index(['BusKey', 'BusID', 'BusType', 'AvgEngineTemp', 'AvgRPM', 'TotalTrips'], dtype='object')
Index(['PerfKey', 'TripID', 'OTPFlag', 'DelayAnalysis', 'AverageSpeed',
       'IdleDuration', 'BrakeEvents'],
      dtype='object')
Index(['RouteKey', 'RouteID', 'RouteName', 'Source', 'Destination',
       'RouteDistanceKM', 'RouteType'],
      dtype='object')
Index(['TimeKey', 'DateKey', 'FullDate', 'Hour', 'DayOfWeek', 'Month',
       'Quarter'],
      dtype='object')


In [20]:
Q1 = fact['Distance'].quantile(0.25)
Q3 = fact['Distance'].quantile(0.75)
IQR = Q3 - Q1

lower_distance = Q1 - 1.5 * IQR
upper_distance = Q3 + 1.5 * IQR

distance_outliers = fact[
    (fact['Distance'] < lower_distance) |
    (fact['Distance'] > upper_distance)
]

print("Distance Outliers:", len(distance_outliers))

Distance Outliers: 0


In [21]:
Q1 = fact['FuelConsumed'].quantile(0.25)
Q3 = fact['FuelConsumed'].quantile(0.75)
IQR = Q3 - Q1

lower_fuel = Q1 - 1.5 * IQR
upper_fuel = Q3 + 1.5 * IQR

fuel_outliers = fact[
    (fact['FuelConsumed'] < lower_fuel) |
    (fact['FuelConsumed'] > upper_fuel)
]

print("FuelConsumed Outliers:", len(fuel_outliers))

FuelConsumed Outliers: 0


In [22]:
print("RouteKey Match:", fact['RouteKey'].isin(route['RouteKey']).all())
print("BusKey Match:", fact['BusKey'].isin(bus['BusKey']).all())
print("TimeKey Match:", fact['TimeKey'].isin(time['TimeKey']).all())
print("PerfKey Match:", fact['PerfKey'].isin(performance['PerfKey']).all())

RouteKey Match: True
BusKey Match: True
TimeKey Match: True
PerfKey Match: True


In [23]:
print("RouteKey Match:", fact['RouteKey'].isin(route['RouteKey']).all())
print("BusKey Match:", fact['BusKey'].isin(bus['BusKey']).all())
print("TimeKey Match:", fact['TimeKey'].isin(time['TimeKey']).all())
print("PerfKey Match:", fact['PerfKey'].isin(performance['PerfKey']).all())

RouteKey Match: True
BusKey Match: True
TimeKey Match: True
PerfKey Match: True


In [24]:
missing_route_keys = fact.loc[
    ~fact['RouteKey'].isin(route['RouteKey']),
    'RouteKey'
].unique()

print("Missing RouteKeys:", missing_route_keys)
print("Count:", len(missing_route_keys))

Missing RouteKeys: []
Count: 0


In [25]:
print(route['RouteKey'].min())
print(route['RouteKey'].max())
print(route['RouteKey'].nunique())

1
150
150


In [26]:
route.shape
route.head()
route.tail()

,RouteKey,RouteID,RouteName,Source,Destination,RouteDistanceKM,RouteType
145,146,R246,Route_246,Warangal,Ramagundam,110,Inter-City
146,147,R247,Route_247,Mahbubnagar,Kodangal,50,Inter-City
147,148,R248,Route_248,Hyderabad,Yellareddy,175,Inter-City
148,149,R249,Route_249,Karimnagar,Korutla,40,Inter-City
149,150,R250,Route_250,Khammam,Suryapet,90,Inter-City


In [27]:
print("RouteKey Match:", fact['RouteKey'].isin(route['RouteKey']).all())
print("BusKey Match:", fact['BusKey'].isin(bus['BusKey']).all())
print("TimeKey Match:", fact['TimeKey'].isin(time['TimeKey']).all())
print("PerfKey Match:", fact['PerfKey'].isin(performance['PerfKey']).all())


RouteKey Match: True
BusKey Match: True
TimeKey Match: True
PerfKey Match: True


In [28]:
merged_df = (
    fact
    .merge(route, on='RouteKey', how='left')
    .merge(bus, on='BusKey', how='left')
    .merge(time, on='TimeKey', how='left')
    .merge(performance, on='PerfKey', how='left')
)

print(merged_df.shape)

(20000, 44)


In [29]:
merged_df.isnull().sum().sum()

np.int64(0)

In [30]:
merged_df.to_csv("cityline_day10_cleaned_merged.csv", index=False)